# M20 lab solution: Multimodal RAG

**Brief.** Chart-aware RAG: index a report that holds key numbers only in figures and show which of the three approaches answers questions about them.

Run with the `colpali` group: `uv run --group colpali jupyter lab`. It provides `transformers` and `torch` for CLIP and `byaldi` for ColPali (the first run downloads both models; ColPali is several GB, and a GPU helps). The chat model must accept images.

Open this notebook from `lab/solution/`, so `../data/` resolves to the lab's own data folder.

## Setup

In [ ]:
import base64
import io
import json
import sys
import tempfile
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
sys.path.insert(0, str(Path("../../../../src").resolve()))

import pymupdf  # noqa: E402
from helpers import get_embeddings, get_experientiallabs_llm  # noqa: E402
from langchain_core.messages import HumanMessage  # noqa: E402
from PIL import Image  # noqa: E402

REPORT = Path("../data/report.pdf")
questions = [json.loads(line) for line in Path("../data/questions.jsonl").read_text().splitlines() if line.strip()]
llm = get_experientiallabs_llm()
embeddings = get_embeddings()
print(len(questions), "questions | LLM:", llm.model_name)

## 1. What the PDF text layer holds

Each page's paragraph is text; each chart is a raster image. The text says what a figure shows but not its numbers, so a text-only index cannot answer these questions at any `k`. The cell extracts both: page text, and every embedded image as a PIL image with its page number.

In [ ]:
pdf = pymupdf.open(REPORT)
pages, figures = [], []
for n, page in enumerate(pdf, 1):
    pages.append({"page": n, "text": page.get_text().strip()})
    for img in page.get_images(full=True):
        pix = pymupdf.Pixmap(pdf, img[0])
        figures.append({"page": n, "image": Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")})
for p in pages:
    print(p["page"], p["text"][:110].replace("\n", " "))
print(len(figures), "figures")


def b64(image):
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()


def ask(question, text=None, image_b64=None):
    content = [{"type": "text", "text": "Answer from the material below only, in one short sentence. "
                                        "If it does not contain the answer, say I don't know.\n\n"
                                        + (f"Text:\n{text}\n\n" if text else "") + f"Question: {question}"}]
    if image_b64:
        content.append({"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}})
    return llm.invoke([HumanMessage(content=content)]).content


def correct(expected, reply):
    """Every comma-separated part of the expected answer appears in the reply (case and thousands separators ignored)."""
    norm = reply.lower().replace(",", "")
    return all(part.strip().lower().replace(",", "") in norm for part in expected.split(", "))

## 2. Approach 1: one CLIP space for text and images

`openai/clip-vit-base-patch32` embeds the page paragraphs and the figures into one space. Each question is embedded as text and the top 3 items are retrieved from the shared ranking. When a figure is among them, it goes to the model with the question; otherwise only the text does.

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")


def features(out):
    # get_*_features returns a tensor in transformers 4 and a model output in some later releases.
    return out if isinstance(out, torch.Tensor) else out.pooler_output


def clip_text(texts_):
    with torch.no_grad():
        v = features(clip.get_text_features(**proc(text=texts_, return_tensors="pt", padding=True, truncation=True)))
    return torch.nn.functional.normalize(v, dim=-1)


def clip_image(images):
    with torch.no_grad():
        v = features(clip.get_image_features(**proc(images=images, return_tensors="pt")))
    return torch.nn.functional.normalize(v, dim=-1)


items = [{"kind": "text", "page": p["page"], "text": p["text"]} for p in pages] + \
        [{"kind": "image", "page": f["page"], "image": f["image"]} for f in figures]
item_vecs = torch.cat([clip_text([p["text"] for p in pages]), clip_image([f["image"] for f in figures])])

clip_rows = []
for q in questions:
    sims = (clip_text([q["question"]]) @ item_vecs.T)[0]
    top = [items[i] for i in sims.argsort(descending=True)[:3].tolist()]
    image = next((t for t in top if t["kind"] == "image"), None)
    text = "\n\n".join(t["text"] for t in top if t["kind"] == "text")
    reply = ask(q["question"], text=text, image_b64=b64(image["image"]) if image else None)
    clip_rows.append({"id": q["id"], "approach": "CLIP shared space", "figure_sent": bool(image),
                      "right_page": bool(image) and image["page"] == q["page"], "correct": correct(q["answer"], reply), "reply": reply})
    print(clip_rows[-1])

## 3. Approach 2: caption, then index text

At index time the vision model writes a caption for each figure, asked to copy every number it can read. The captions and the page paragraphs are embedded with the course embeddings and searched as text (top 3); the model answers from text only.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

CAPTION = "Describe this chart for a search index: its title, axes, and every label and value printed on it."
captions = []
for f in figures:
    msg = HumanMessage(content=[{"type": "text", "text": CAPTION},
                                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64(f['image'])}"}}])
    captions.append(Document(page_content=llm.invoke([msg]).content, metadata={"page": f["page"], "kind": "caption"}))
    print(f["page"], captions[-1].page_content[:150].replace("\n", " "))

text_store = FAISS.from_documents(captions + [Document(page_content=p["text"], metadata={"page": p["page"], "kind": "text"})
                                              for p in pages], embeddings)
caption_rows = []
for q in questions:
    docs = text_store.similarity_search(q["question"], k=3)
    reply = ask(q["question"], text="\n\n".join(d.page_content for d in docs))
    caption_rows.append({"id": q["id"], "approach": "caption then index", "figure_sent": False,
                         "right_page": any(d.metadata["kind"] == "caption" and d.metadata["page"] == q["page"] for d in docs),
                         "correct": correct(q["answer"], reply), "reply": reply})
    print(caption_rows[-1])

## 4. Approach 3: page-image retrieval with ColPali

`byaldi` indexes each page as an image with `vidore/colpali-v1.2` (the model of the module's ColPali notebook) and scores pages against the question by late interaction. The top page image goes to the model with the question. The index is written to a temporary folder.

In [ ]:
from byaldi import RAGMultiModalModel

index_root = tempfile.mkdtemp(prefix="m20_byaldi_")
colpali = RAGMultiModalModel.from_pretrained("vidore/colpali-v1.2", index_root=index_root, verbose=0)
colpali.index(input_path=str(REPORT), index_name="larkspur_report", store_collection_with_index=True, overwrite=True)

colpali_rows = []
for q in questions:
    (hit,) = colpali.search(q["question"], k=1)
    reply = ask(q["question"], image_b64=hit.base64)
    colpali_rows.append({"id": q["id"], "approach": "ColPali page image", "figure_sent": True,
                         "right_page": hit.page_num == q["page"], "correct": correct(q["answer"], reply), "reply": reply})
    print(colpali_rows[-1])

## Results: which approach answers which question

In [ ]:
import pandas as pd

df = pd.DataFrame(clip_rows + caption_rows + colpali_rows)
print("run:", time.strftime("%Y-%m-%d"), "| LLM:", llm.model_name)
print(df.pivot(index="id", columns="approach", values="correct"))
print(df.groupby("approach")[["right_page", "correct"]].sum())

## What this shows

- The PDF text layer names each figure but holds none of its numbers, so all three approaches have to get a chart image, or a description of one, in front of the model.
- In one CLIP ranking, text and images score on different scales, so a figure can be the best image for a question and still fall outside the top 3. The `figure_sent` column shows how often that happened.
- Captioning moves the chart reading to index time: the answers are only as good as the numbers the caption wrote down. Read the printed captions.
- Page-image retrieval keeps the chart, its labels and the paragraph together and sends the page itself, at the cost of a large model and many vectors per page.